# Operational Routes Testing

This notebook tests the operational cache management endpoints of the Bedrock Proxy Gateway.


In [ ]:
import json

import boto3
import requests



import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment-specific .env file (e.g., .env.dev, .env.test)
ENVIRONMENT = os.getenv("ENVIRONMENT", "dev")
env_file = Path(__file__).parent / f".env.{ENVIRONMENT}" if "__file__" in dir() else Path(f".env.{ENVIRONMENT}")
load_dotenv(env_file)

# Configuration
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_URL = os.getenv("GATEWAY_API_URL", "https://your-gateway-url.com")
SECRET_ID = os.getenv("GATEWAY_SECRET_ID", "bedrock-gateway-dev-oauth-credentials")

# Extract the secret value from Secret Manager

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name=AWS_REGION)
JSON_SECRET = json.loads(session.client('secretsmanager').get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

# Token Generation


In [ ]:
"""Generate access token for API authentication"""
payload = {
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "grant_type": "client_credentials",
    "audience": "bedrockproxygateway",
    "scope": "bedrockproxygateway:admin"
}

response = requests.post(TOKEN_URL, data=payload)
response.raise_for_status()
TOKEN = response.json()['access_token']

# List Cache Keys

List all cache keys accessible to the current client.


In [ ]:
def list_cache_keys():
    """List all cache keys."""
    url = f"{API_URL}/cache/keys"
    headers = {"Authorization": f"Bearer {TOKEN}"}
    response = requests.get(url, headers=headers, verify=False)
    response.raise_for_status()
    return response.json()

result = list_cache_keys()
print(json.dumps(result, indent=2))

# Clear Cache by Key

Clear a specific cache key.


In [ ]:
def clear_cache_by_key(cache_key: str):
    """Clear a specific cache key."""
    url = f"{API_URL}/cache/keys/{cache_key}"
    headers = {"Authorization": f"Bearer {TOKEN}"}
    response = requests.delete(url, headers=headers)
    response.raise_for_status()
    return response.json()

# Example: Clear a specific key (replace with actual key from list_cache_keys)
# cache_key = "quota:CCMCKKPFTOCVXTZEHMYTLGDLCOPAPNHQ:us.amazon.nova-lite-v1:0"
# result = clear_cache_by_key(cache_key)
# print(json.dumps(result, indent=2))

print("Uncomment and set cache_key to test")

# Clear All Cache Keys

Clear all cache keys accessible to the current client.


In [ ]:
def clear_all_cache_keys():
    """Clear all cache keys."""
    url = f"{API_URL}/cache/keys"
    headers = {"Authorization": f"Bearer {TOKEN}"}
    response = requests.delete(url, headers=headers)
    response.raise_for_status()
    return response.json()

# Uncomment to test clearing all keys
# result = clear_all_cache_keys()
# print(json.dumps(result, indent=2))

print("Uncomment to test clearing all cache keys")

# Complete Test Workflow

Test the complete workflow: list keys, clear specific key, list again, clear all.


In [ ]:
print("=== Step 1: List initial cache keys ===")
initial_keys = list_cache_keys()
print(f"Client ID: {initial_keys['client_id']}")
print(f"Is Admin: {initial_keys['is_admin']}")
print(f"Total keys: {initial_keys['count']}")
if initial_keys['keys']:
    print(f"First few keys: {initial_keys['keys'][:3]}")

# Only proceed if there are keys to clear
if initial_keys['count'] > 0:
    print("\n=== Step 2: Clear a specific key ===")
    first_key = initial_keys['keys'][0]
    print(f"Clearing key: {first_key}")
    clear_result = clear_cache_by_key(first_key)
    print(json.dumps(clear_result, indent=2))
    print("\n=== Step 3: List keys after clearing one ===")
    after_clear_keys = list_cache_keys()
    print(f"Total keys: {after_clear_keys['count']}")
    print(f"Keys removed: {initial_keys['count'] - after_clear_keys['count']}")
else:
    print("\nNo keys to clear. Cache is empty.")

print("\n=== Test workflow completed ===")

# Admin Privileges Test

Test admin privileges (only works if CLIENT_ID is admin).


In [ ]:
print("=== Admin Privileges Test ===")
keys_result = list_cache_keys()

if keys_result['is_admin']:
    print("You have admin privileges!")
    print(f"You can see all {keys_result['count']} cache keys in the system")
    print("Admin can clear any key or all keys without restrictions")
else:
    print("You have regular user privileges")
    print(f"You can see {keys_result['count']} cache keys belonging to your client_id")
    print("You can only clear your own cache keys")

print(f"\nClient ID: {keys_result['client_id']}")